# Ansible vs Terraform for infrastructure provisioning: decision matrix

A structured comparison to choose between configuration-management-driven and declarative-state-driven provisioning for cloud and on-prem infrastructure.

> **Scope** — written against ansible-core 2.21.5 (published 2026-10-05, current GA line). Terraform references reflect the 1.9+ feature line; specific version claims for Terraform are omitted pending research corpus coverage.

---

**last_verified:** 2026-10-20  
**tool_version:** ansible-core 2.21.5  
**sources:** https://api.github.com/repos/ansible/ansible/releases?per_page=3 https://docs.ansible.com/projects/ansible-core/devel/roadmap/ROADMAP_2_22.html https://raw.githubusercontent.com/ansible/ansible/v2.22.0b2/changelogs/CHANGELOG-v2.22.rst https://api.github.com/repos/ansible/awx-operator/releases/latest

## Purpose

Infrastructure provisioning sits at the intersection of desired-state declaration and imperative execution. Ansible approaches it through configuration management — describing tasks that converge a system toward a target state. Terraform approaches it through declarative infrastructure-as-code — describing the desired resource graph and relying on a provider-driven planner to compute and apply changes. This notebook maps the two approaches across the axes that drive tool selection: lifecycle ownership, state management, drift handling, team workflow, and ecosystem fit.

## When to use

- **Choose Ansible** when the provisioning workflow is tightly coupled to configuration management (packages, services, files), when the team already standardizes on Ansible for day-two operations, when ad-hoc imperative steps are frequent, or when the target environment includes heterogeneous systems (network devices, Windows, legacy platforms) that lack Terraform providers.
- **Choose Terraform** when the primary concern is cloud resource lifecycle (create/update/destroy of VPCs, compute, databases, IAM), when a single source of truth for infrastructure state is required, when plan/apply review gates are a compliance requirement, or when the team needs strong dependency ordering and parallelism across large resource graphs.
- **Use both** when cloud provisioning (Terraform) hands off to configuration management (Ansible) — the dominant hybrid pattern. Terraform creates the substrate; Ansible configures the workload.

## Prerequisites

- ansible-core 2.21.x installed (pin 2.21.5 for production; 2.22 pre-releases in staging until GA).
- Terraform CLI installed (version aligned with provider requirements; no specific version mandated here due to research corpus scope).
- Access to a target cloud provider (AWS, Azure, GCP) or on-prem platform with corresponding Terraform providers and Ansible collections.
- Version control for both Terraform configurations and Ansible playbooks/roles.
- For hybrid workflows: a mechanism to pass Terraform outputs (resource IDs, endpoints, credentials) into Ansible inventory or playbook variables.

## 1. Provisioning philosophy comparison

Ansible models provisioning as a sequence of tasks executed against an inventory. Each task declares a desired state for a specific resource (package installed, service started, file present). The playbook runs top to bottom; idempotency is the responsibility of each module.

Terraform models provisioning as a resource graph. The configuration declares resources and their relationships; the planner builds a dependency graph, computes a plan (create/update/destroy actions), and applies it. State is stored in a backend (local, remote, or Terraform Cloud) and used to detect drift on subsequent runs.

In [ ]:
# Ansible provisioning sketch — tasks converge toward state
cat > /tmp/ansible-provision.yml <<'EOF'
---
- name: Provision web tier
  hosts: localhost
  connection: local
  tasks:
    - name: Create VPC
      amazon.aws.ec2_vpc_net:
        name: web-vpc
        cidr_block: 10.0.0.0/16
        region: us-east-1
      register: vpc
    - name: Create subnet
      amazon.aws.ec2_vpc_subnet:
        vpc_id: "{{ vpc.vpc.id }}"
        cidr: 10.0.1.0/24
        region: us-east-1
      register: subnet
    - name: Launch instance
      amazon.aws.ec2_instance:
        vpc_subnet_id: "{{ subnet.subnet.id }}"
        image_id: ami-0abcdef1234567890
        instance_type: t3.micro
        region: us-east-1
EOF
echo "ansible provisioning sketch written to /tmp/ansible-provision.yml"
cat /tmp/ansible-provision.yml

## 2. Terraform provisioning sketch — resources declare desired graph

Terraform configurations declare resources with implicit dependencies. The planner orders operations automatically.

In [ ]:
cat > /tmp/terraform-provision.tf <<'EOF'
# Terraform provisioning sketch — resource graph
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  region = "us-east-1"
}

resource "aws_vpc" "web" {
  cidr_block = "10.0.0.0/16"
  tags = { Name = "web-vpc" }
}

resource "aws_subnet" "web" {
  vpc_id     = aws_vpc.web.id
  cidr_block = "10.0.1.0/24"
  tags = { Name = "web-subnet" }
}

resource "aws_instance" "web" {
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.micro"
  subnet_id     = aws_subnet.web.id
  tags = { Name = "web-instance" }
}
EOF
echo "terraform provisioning sketch written to /tmp/terraform-provision.tf"
cat /tmp/terraform-provision.tf

## 3. Decision matrix

| Dimension | Ansible (configuration management) | Terraform (declarative IaC) |
|---|---|---|
| **Primary abstraction** | Tasks → plays → playbooks | Resources → modules → configurations |
| **State management** | Implicit (target system is source of truth) | Explicit (state file/backend tracks resource IDs) |
| **Drift detection** | Re-run playbook with `--check`; modules report changes | `terraform plan` compares state backend vs real infrastructure |
| **Dependency ordering** | Explicit in playbook (task order, `depends_on` in roles) | Automatic from resource references (implicit graph) |
| **Parallelism** | `serial`, `forks`, `strategy: free` | Automatic graph-based parallelism |
| **Secrets handling** | Vault, `no_log`, vars plugins; secrets in task output unless masked | State may contain sensitive values; encryption at rest via backend; `sensitive` flag on outputs |
| **Day-two operations** | Native strength — packages, services, files, users | Limited to resource recreation; `null_resource` + provisioners as escape hatch |
| **Heterogeneous targets** | Strong — network, Windows, storage, containers via collections | Provider-dependent; gaps for non-cloud platforms |
| **Policy/compliance gates** | AAP job templates, ansible-policy, `ansible-lint` | `terraform plan` review, Sentinel/OPA policies, Terraform Cloud run tasks |
| **Learning curve** | Lower for ops teams familiar with SSH/task automation | Higher — requires understanding of state, providers, HCL |
| **Rollback** | Re-run previous playbook version; no built-in state rollback | `terraform apply` previous state version (if backend supports versioning) |
| **Multi-cloud** | Via collections per cloud; same task syntax | Via providers per cloud; same resource syntax pattern |
| **Ecosystem maturity (provisioning)** | Cloud modules exist (amazon.aws, azure.azcollection, google.cloud) but are task-oriented | Native provider resources for 3000+ resource types across major clouds |


## 4. Hybrid pattern: Terraform provisions, Ansible configures

The most common production pattern combines both tools. Terraform manages the cloud resource lifecycle (VPC, subnets, load balancers, databases, IAM). Ansible manages the workload configuration (OS hardening, application deploy, service mesh sidecars, certificate rotation). The handoff uses Terraform outputs as Ansible inventory sources or playbook variables.

In [ ]:
# Hybrid handoff sketch — Terraform outputs → Ansible inventory
cat > /tmp/hybrid-outputs.tf <<'EOF'
output "web_vpc_id" {
  value       = aws_vpc.web.id
  description = "VPC ID for Ansible inventory"
}

output "web_subnet_ids" {
  value       = aws_subnet.web[*].id
  description = "Subnet IDs for Ansible inventory"
}

output "web_instance_ips" {
  value       = aws_instance.web[*].private_ip
  description = "Instance IPs for Ansible inventory"
  sensitive   = false
}
EOF

cat > /tmp/ansible-inventory-from-tf.yml <<'EOF'
---
# Dynamic inventory built from Terraform outputs (conceptual)
plugin: constructed
strict: false
groups:
  web_tier: "'web' in (tags.Name | default(''))"
compose:
  ansible_host: private_ip_address
  ansible_user: ec2-user
EOF
echo "hybrid handoff sketches written"

## 5. Ansible-specific provisioning enhancements (2.21/2.22 line)

- **ansible.platform collection** (AAP 2.7 GA): unified controller-side CaC interface (22 modules + 1 lookup) running as action plugins on the controller node. Tasks target `localhost` with `connection: local` or `connection: ansible.platform.http`. Enables declarative AAP resource management (inventories, credentials, job templates) from playbooks.
- **Vault OIDC (Tech Preview, AAP 2.7)**: AAP acts as native OIDC identity provider for HashiCorp Vault. Jobs authenticate with short-lived JWTs scoped to job/user/org. New credential types: HashiCorp Vault Secret Lookup (OIDC) and HashiCorp Vault Signed SSH (OIDC).
- **Execution environments**: Containerized runtime keeps provisioning collections (amazon.aws, azure.azcollection, google.cloud, community.kubernetes) portable across laptop, CI, and AAP.
- **Collections for cloud**: amazon.aws (EC2, VPC, RDS, IAM), azure.azcollection (resource groups, VMs, AKS), google.cloud (GCE, GKE, Cloud SQL). Versioned independently of ansible-core.

## 6. Terraform-specific provisioning enhancements (1.9+ line)

- **Ephemeral values** (1.9+): declare values that exist only during plan/apply (e.g., generated passwords, tokens) without persisting to state.
- **Provider-defined functions** (1.9+): providers can expose custom functions for use in expressions (e.g., cloud-specific ID parsing).
- **Moved blocks** (1.9+): refactor resource addresses in state without destruction (`moved { from = old; to = new }`).
- **Terraform Cloud/Enterprise**: remote runs, policy-as-code (Sentinel/OPA), drift detection, private module registry, run tasks for third-party integration.
- **Testing**: `terraform test` (1.6+) for unit/integration tests against real or mocked providers.

## Verify

1. **Decision helper**: run the scenario classifier below — each input resolves to a recommended primary tool or the hybrid pattern.
2. **Plan parity**: for a given infrastructure design, produce both an Ansible playbook and a Terraform configuration. Apply each to a test environment and compare resulting resource sets.
3. **Drift detection**: introduce manual changes in the test environment. Run `ansible-playbook --check` and `terraform plan` — both must surface the drift.
4. **Handoff test**: apply the Terraform sketch, emit outputs, generate an Ansible inventory from those outputs, and run a configuration playbook against it. Verify the workload deploys correctly.
5. **Version pins**: confirm ansible-core reports 2.21.x on production paths; Terraform CLI version matches provider version constraints in the configuration.

In [ ]:
pick_tool() {
    case "$1" in
        cloud-resources|vpc|iam|database|lb)       echo "Terraform: native provider resources, stateful lifecycle, plan gate" ;;
        config-mgmt|packages|services|files|users) echo "Ansible: idempotent modules, day-two strength, heterogeneous targets" ;;
        hybrid|full-stack|handoff)                  echo "Hybrid: Terraform provisions substrate, Ansible configures workload" ;;
        network-device|windows|legacy)              echo "Ansible: collection coverage where Terraform providers do not exist" ;;
        policy-gate|compliance|audit)               echo "Both: Terraform plan review + Sentinel/OPA; AAP job templates + ansible-policy" ;;
        *)                                          echo "Default to Terraform for cloud resources; Ansible for configuration" ;;
    esac
}
for s in cloud-resources config-mgmt hybrid network-device policy-gate unknown; do
    printf '%-18s -> %s\n' "$s" "$(pick_tool "$s")"
done

## Common errors

- **Managing cloud resources solely via Ansible when Terraform providers exist**: task-based cloud modules lack the dependency graph, plan review, and state-backed drift detection that Terraform provides. Result: implicit ordering bugs, missed parallelism, no audit trail of infrastructure changes.
- **Using Terraform provisioners for configuration management**: `remote-exec` and `local-exec` provisioners are an anti-pattern for day-two work. They bypass Terraform's declarative model, lack idempotency guarantees, and couple configuration logic to the apply cycle. Hand off to Ansible instead.
- **State drift undetected**: skipping `terraform plan` in CI/CD or running Ansible playbooks without `--check` before apply. Both tools require the dry-run step to catch drift early.
- **Secrets in state or output**: Terraform state may contain sensitive values (RDS passwords, TLS private keys). Use ephemeral values (1.9+) or external secrets stores. Ansible `no_log` masks task output but does not prevent secrets in callback plugins or AWX job logs without additional redaction.
- **Version skew between collections and providers**: amazon.aws collection version vs hashicorp/aws provider version may support different API surfaces. Pin both in CI and test the combination.
- **Split-brain hybrid handoff**: Terraform outputs change but Ansible inventory is not regenerated. Automate inventory generation as a post-apply step or use a dynamic inventory plugin that queries the cloud API directly.

## References

- Ansible current GA and stable lines (2.21.5, 2.20.10, 2.19.14, all 2026-10-05): https://api.github.com/repos/ansible/ansible/releases?per_page=3 
- Ansible 2.22 release-phase schedule and planned/delayed work: https://docs.ansible.com/projects/ansible-core/devel/roadmap/ROADMAP_2_22.html 
- Ansible 2.22 breaking changes and deprecation watch-list (incl. CVE-2026-11332 identifier; the changelog publishes no CVSS score or version range for it): https://raw.githubusercontent.com/ansible/ansible/v2.22.0b2/changelogs/CHANGELOG-v2.22.rst 
- AWX operator release state (2.19.1 with AWX v24.6.1): https://api.github.com/repos/ansible/awx-operator/releases/latest 
- Red Hat Ansible Automation Platform 2.7 new features (automation dashboard, EE builder, content catalog, orchestrator, MCP server, Lightspeed, ansible.platform): https://developers.redhat.com/articles/2026/06/10/whats-new-red-hat-ansible-automation-platform-2-7